# Titanic Survival Analysis

End-to-end exploratory analysis and classification notebook for the classic Titanic dataset.

This notebook covers data loading, cleaning, feature engineering, visualization, and baseline machine learning models to study which passenger attributes were associated with survival.

## 1. Setup

Import the libraries and make the project root available so we can reuse the helper functions from `src/`.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.preprocessing import prepare_dataset
from src.modeling import split_data, train_models, evaluate_model

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (10, 6)


## 2. Load the data

The notebook expects the Kaggle Titanic training file at `data/raw/train.csv`. If you want to use the same project structure from this repository, place the file there before running the notebook.

In [ ]:
data_path = PROJECT_ROOT / 'data' / 'raw' / 'train.csv'

df = pd.read_csv(data_path)
df.head()


## 3. Quick data audit

Look at the shape, column types, and missing values before cleaning.

In [ ]:
display(df.shape)
display(df.info())
display(df.describe(include='all').T)

missing = df.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0].to_frame(name='missing_values')
display(missing)


## 4. Cleaning and feature engineering

Standardize the dataset, create derived features, and keep the target variable intact.

In [ ]:
clean_df = prepare_dataset(df)

# Extra features that are useful in the Titanic problem
clean_df['is_alone'] = (clean_df['family_size'] == 1).astype(int)
clean_df['fare_log'] = np.log1p(clean_df['fare'])

display(clean_df.head())
display(clean_df.isna().sum().sort_values(ascending=False).head(10))


## 5. Target distribution

Check the balance of the survival target. The Titanic dataset is a binary classification problem: `survived = 0` means the passenger did not survive and `survived = 1` means the passenger survived.

In [ ]:
survival_rate = clean_df['survived'].mean()
print(f'Survival rate: {survival_rate:.2%}')

sns.countplot(data=clean_df, x='survived', palette='Set2')
plt.title('Survival distribution')
plt.xlabel('Survived')
plt.ylabel('Count')
plt.show()


## 6. Survival patterns

Explore how survival changes across sex, passenger class, embarkation point, and family structure.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

sns.barplot(data=clean_df, x='sex', y='survived', ax=axes[0, 0], palette='Set2')
axes[0, 0].set_title('Survival rate by sex')
axes[0, 0].set_xlabel('Sex')
axes[0, 0].set_ylabel('Survival rate')

sns.barplot(data=clean_df, x='pclass', y='survived', ax=axes[0, 1], palette='Set2')
axes[0, 1].set_title('Survival rate by passenger class')
axes[0, 1].set_xlabel('Passenger class')
axes[0, 1].set_ylabel('Survival rate')

sns.barplot(data=clean_df, x='embarked', y='survived', ax=axes[1, 0], palette='Set2')
axes[1, 0].set_title('Survival rate by embarkation port')
axes[1, 0].set_xlabel('Embarked')
axes[1, 0].set_ylabel('Survival rate')

sns.barplot(data=clean_df, x='is_alone', y='survived', ax=axes[1, 1], palette='Set2')
axes[1, 1].set_title('Survival rate by travel status')
axes[1, 1].set_xlabel('Is alone (1 = yes)')
axes[1, 1].set_ylabel('Survival rate')

plt.tight_layout()
plt.show()


## 7. Age and fare analysis

Look at the distribution of age and fare and compare them across survival outcomes.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

sns.histplot(data=clean_df, x='age', hue='survived', kde=True, bins=30, ax=axes[0, 0])
axes[0, 0].set_title('Age distribution by survival')

sns.boxplot(data=clean_df, x='survived', y='age', ax=axes[0, 1], palette='Set3')
axes[0, 1].set_title('Age by survival')
axes[0, 1].set_xlabel('Survived')

sns.boxplot(data=clean_df, x='survived', y='fare', ax=axes[1, 0], palette='Set3')
axes[1, 0].set_title('Fare by survival')
axes[1, 0].set_xlabel('Survived')

sns.scatterplot(data=clean_df, x='age', y='fare', hue='survived', alpha=0.6, ax=axes[1, 1])
axes[1, 1].set_title('Age vs fare')

plt.tight_layout()
plt.show()


## 8. Correlation on numeric features

A quick numeric correlation view helps identify linear relationships among the main quantitative variables.

In [ ]:
numeric_cols = ['survived', 'pclass', 'age', 'sibsp', 'parch', 'fare', 'family_size', 'fare_log', 'is_alone']
corr = clean_df[numeric_cols].corr(numeric_only=True)

plt.figure(figsize=(12, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', square=True)
plt.title('Correlation heatmap')
plt.show()


## 9. Modeling dataset

Select the most useful features for classification. The target remains `survived`.

In [ ]:
feature_cols = [
    'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked',
    'title', 'family_size', 'age_group', 'is_alone', 'fare_log'
]

feature_cols = [col for col in feature_cols if col in clean_df.columns]
model_df = clean_df[feature_cols + ['survived']].copy()

display(model_df.head())
display(model_df.dtypes)


## 10. Train and evaluate baseline models

Train two standard classifiers: Logistic Regression and Random Forest.

In [ ]:
X_train, X_test, y_train, y_test = split_data(model_df, target_col='survived', test_size=0.2)
pipelines = train_models(X_train, y_train)

results = []
for model_name, pipeline in pipelines.items():
    result = evaluate_model(pipeline, X_test, y_test)
    results.append({
        'model': model_name,
        'accuracy': result.accuracy,
        'report': result.report,
        'confusion': result.confusion,
    })

results_df = pd.DataFrame(results).sort_values('accuracy', ascending=False).reset_index(drop=True)
display(results_df[['model', 'accuracy']])
display(results_df['report'].iloc[0])


## 11. Best model diagnostics

Visualize the confusion matrix for the best-performing model.

In [ ]:
best_model_name = results_df.loc[0, 'model']
best_pipeline = pipelines[best_model_name]
best_result = evaluate_model(best_pipeline, X_test, y_test)

conf_matrix = np.array(best_result.confusion)
plt.figure(figsize=(6, 5))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title(f'Confusion matrix: {best_model_name}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

print(f'Best model: {best_model_name}')
print(f'Accuracy: {best_result.accuracy:.4f}')
print(best_result.report)


## 12. Conclusions

Main takeaways to highlight in the portfolio version of this project:

- Survival is strongly associated with passenger sex and class.
- Family structure, fare, and age add useful context to the analysis.
- Feature engineering improves the clarity of the data and can help baseline models.
- Logistic Regression and Random Forest are solid first models for a junior portfolio project.

Next steps:

- Tune hyperparameters and test cross-validation.
- Add a cleaner preprocessing pipeline with `ColumnTransformer` and `Pipeline`.
- Export charts to `reports/figures/`.
- Write a short project summary for LinkedIn and the GitHub README.